# Debug: Training Pipeline Visualisation

This notebook visualises:
1. SA-1B sample loading & annotation filtering
2. Puzzles box generation with overlap analysis
3. **1:1 Prompt-GT correspondence** — `prompt[:, i]` and `gt[:, i]` are the SAME object
4. **Per-instance mapping** — prompt on 1 view, GT on ALL views
5. **Data augmentation** — geometric, pixel, and occlusion effects
6. **Curriculum scheduling** — overlap parameter progression (easy+hard mix)
7. Multi-sample browsing
8. Pipeline feature status summary

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(".").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
sys.path.insert(0, str(PROJECT_ROOT / "repo" / "Pi3"))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
import torch

from sa1b.dataset import SA1BReader
from sa1b.sa1b_puzzles_dataset import SA1BPuzzlesDataset
from sa1b.augmentations import CropAugmentor
from sa1b.visualization import decode_segmentation
from sa1b.puzzles_bridge import generate_puzzle_crops

%matplotlib inline
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (14, 6)

## 1. Load an SA-1B sample

In [ ]:
SA1B_ROOT = "/mnt/dataset/marko/data/sa1b/data"
reader = SA1BReader(SA1B_ROOT)
shards = reader.list_shards()
print(f"Available shards: {len(shards)}")

shard = shards[0]
stems = reader.list_sample_stems(shard)
print(f"Shard '{shard}': {len(stems)} samples")

sample = reader.load_sample(shard, stems[0])
img_bgr = cv2.imread(str(sample.image_path))
print(f"Image shape: {img_bgr.shape}")
print(f"Annotations: {len(sample.annotations)}")

fig, ax = plt.subplots(1, 1, figsize=(8, 6))
ax.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
ax.set_title(f"SA-1B sample: {stems[0]}")
ax.axis("off")
plt.tight_layout()
plt.show()

## 2. Puzzles box generation & overlap analysis

In [ ]:
puzzle_result = generate_puzzle_crops(
    sample, seed=42, n_boxes=6,
    min_box_size_scale=0.45, max_box_size_scale=0.9,
    min_overlap=0.1, max_overlap=0.3,
    min_mask_coverage=0.5, max_annotations=120,
    min_area=1024, min_predicted_iou=0.85,
    min_stability_score=0.9,
)

boxes = puzzle_result.boxes
img_h, img_w = puzzle_result.image.shape[:2]
print(f"Generated {len(boxes)} boxes (image {img_w}x{img_h})")

# Compute per-box scale
for i, (x1, y1, x2, y2) in enumerate(boxes):
    bw, bh = x2 - x1, y2 - y1
    sx, sy = bw / img_w, bh / img_h
    print(f"  Box {i}: {bw}x{bh}  scale_w={sx:.2f}  scale_h={sy:.2f}")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
ax.imshow(puzzle_result.image)
colors = plt.cm.tab10(np.linspace(0, 1, len(boxes)))
for i, (x1, y1, x2, y2) in enumerate(boxes):
    rect = mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                               linewidth=2, edgecolor=colors[i],
                               facecolor="none", label=f"View {i}")
    ax.add_patch(rect)
ax.legend(fontsize=8)
ax.set_title(f"Puzzles Boxes (scale 0.45~0.9)")
ax.axis("off")

# Pairwise IoU matrix
N = len(boxes)
iou_matrix = np.zeros((N, N))
for i in range(N):
    for j in range(N):
        x1_i, y1_i, x2_i, y2_i = boxes[i]
        x1_j, y1_j, x2_j, y2_j = boxes[j]
        inter_x1 = max(x1_i, x1_j)
        inter_y1 = max(y1_i, y1_j)
        inter_x2 = min(x2_i, x2_j)
        inter_y2 = min(y2_i, y2_j)
        inter = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)
        area_i = (x2_i - x1_i) * (y2_i - y1_i)
        area_j = (x2_j - x1_j) * (y2_j - y1_j)
        union = area_i + area_j - inter
        iou_matrix[i, j] = inter / max(union, 1)

ax = axes[1]
im = ax.imshow(iou_matrix, cmap="YlOrRd", vmin=0, vmax=1)
for i in range(N):
    for j in range(N):
        ax.text(j, i, f"{iou_matrix[i, j]:.2f}", ha="center", va="center", fontsize=8)
ax.set_xlabel("View")
ax.set_ylabel("View")
ax.set_title("Pairwise Box IoU")
plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()


## 3. Dataset Output & 1:1 Prompt-GT Correspondence

Key design:
- `prompt_masks[:, i]` and `gt_consistent_masks[:, i]` refer to the **same** physical object
- prompt is non-zero only on the prompt view; GT is non-zero on **all** views
- If M_prompt = 2, then GT also has exactly 2 channels — no ambiguity
- No Hungarian matching needed — direct channel-wise loss

In [ ]:
ds = SA1BPuzzlesDataset(
    sa1b_root=SA1B_ROOT,
    shard_names=[shards[0]],
    n_views=6,
    resolution=224,
    max_prompts=4,
    augment=False,
    seed=42,
)

item = ds[0]
imgs = item["imgs"]
gt = item["gt_consistent_masks"]
pm = item["prompt_masks"]
pv = item["prompt_views"]

print("=== Dataset output ===")
print(f"  imgs:          {imgs.shape}")
print(f"  prompt_masks:  {pm.shape}")
print(f"  gt_masks:      {gt.shape}")
print(f"  prompt_views:  {pv.numpy().tolist()}")
print()
print(f"  M_prompt = {pm.shape[1]},  M_gt = {gt.shape[1]}  <- MUST be equal (1:1)")
assert pm.shape[1] == gt.shape[1], "BUG: prompt and GT channel count mismatch!"
print("  Assertion passed: prompt_masks[:, i] <-> gt_masks[:, i] are 1:1")
print()
print(f"  box_size_scale: [{ds.puzzle_kwargs['min_box_size_scale']:.2f}, {ds.puzzle_kwargs['max_box_size_scale']:.2f}]")
print(f"  overlap:        [{ds.puzzle_kwargs['min_overlap']:.2f}, {ds.puzzle_kwargs['max_overlap']:.2f}]")


## 4. Per-Instance Visualisation: Prompt ↔ GT 1:1 Mapping

For each prompted instance, we show:
- **Row A (orange)**: Prompt mask — only on the prompt view, zero everywhere else
- **Row B (blue)**: GT mask — the SAME object, but present on ALL views

Channel index `i` in prompt and GT always refers to the same physical object.

In [ ]:
N = imgs.shape[0]
M = pm.shape[1]  # == gt.shape[1], guaranteed 1:1

PALETTE = [
    (0.90, 0.30, 0.30), (0.30, 0.80, 0.30), (0.30, 0.50, 0.90),
    (0.95, 0.75, 0.20), (0.80, 0.30, 0.80), (0.20, 0.85, 0.85),
]

# --- Overview: images + all prompts + all GT ---
fig, axes = plt.subplots(3, N, figsize=(3 * N, 9))
pv_set = set(pv.numpy().tolist())

for v in range(N):
    img_np = imgs[v].permute(1, 2, 0).numpy()

    axes[0, v].imshow(img_np)
    title = f"View {v}"
    if v in pv_set:
        title += " *"
        for spine in axes[0, v].spines.values():
            spine.set_edgecolor("red")
            spine.set_linewidth(3)
    axes[0, v].set_title(title, fontsize=10)
    axes[0, v].axis("off")

    # Prompt overlay
    pm_ov = img_np.copy()
    for m in range(M):
        mask = pm[v, m].numpy() > 0.5
        if mask.any():
            c = np.array(PALETTE[m % len(PALETTE)])
            pm_ov[mask] = pm_ov[mask] * 0.3 + c * 0.7
    axes[1, v].imshow(pm_ov)
    has_prompt = pm[v].sum().item() > 0
    axes[1, v].set_title("Prompt (has)" if has_prompt else "Prompt (empty)", fontsize=9)
    axes[1, v].axis("off")

    # GT overlay (same colors = same instance)
    gt_ov = img_np.copy()
    for m in range(M):
        mask = gt[v, m].numpy() > 0.5
        if mask.any():
            c = np.array(PALETTE[m % len(PALETTE)])
            gt_ov[mask] = gt_ov[mask] * 0.4 + c * 0.6
    axes[2, v].imshow(gt_ov)
    axes[2, v].set_title(f"GT ({M} inst)", fontsize=9)
    axes[2, v].axis("off")

axes[0, 0].set_ylabel("Image", fontsize=10)
axes[1, 0].set_ylabel("Prompt\n(1 view only)", fontsize=10)
axes[2, 0].set_ylabel("GT\n(all views)", fontsize=10)
plt.suptitle(
    f"Overview: {M} prompts, same color = same instance in prompt & GT",
    fontsize=12, fontweight="bold", y=1.02,
)
plt.tight_layout()
plt.show()

# --- Per-instance detail: prompt vs GT across views ---
M_show = min(M, 4)
fig, axes = plt.subplots(M_show * 2, N, figsize=(2.5 * N, M_show * 4))
if M_show == 1 and N > 1:
    axes = axes.reshape(2, N)

for inst in range(M_show):
    prompt_v = int(pv[inst].item())
    color = PALETTE[inst % len(PALETTE)]

    for v in range(N):
        # Prompt row
        pm_val = pm[v, inst].numpy()
        axes[inst * 2, v].imshow(pm_val, cmap="Oranges", vmin=0, vmax=1)
        lbl = f"* V{v}" if v == prompt_v else f"V{v} (0)"
        axes[inst * 2, v].set_title(lbl, fontsize=8)
        axes[inst * 2, v].axis("off")

        # GT row (same instance index)
        gt_val = gt[v, inst].numpy()
        axes[inst * 2 + 1, v].imshow(gt_val, cmap="Blues", vmin=0, vmax=1)
        pct = gt_val.mean() * 100
        axes[inst * 2 + 1, v].set_title(f"{pct:.1f}%", fontsize=8)
        axes[inst * 2 + 1, v].axis("off")

    axes[inst * 2, 0].set_ylabel(f"Prompt #{inst}\n(view {prompt_v})", fontsize=9, color="darkorange")
    axes[inst * 2 + 1, 0].set_ylabel(f"GT #{inst}\n(all views)", fontsize=9, color="steelblue")

plt.suptitle(
    "1:1 Correspondence: prompt[i] and gt[i] are the SAME object",
    fontsize=12, fontweight="bold",
)
plt.tight_layout()
plt.show()

print("\n=== Verification ===")
for inst in range(M_show):
    pv_i = int(pv[inst].item())
    pm_pixels = (pm[pv_i, inst] > 0.5).sum().item()
    gt_pixels = (gt[pv_i, inst] > 0.5).sum().item()
    overlap = ((pm[pv_i, inst] > 0.5) & (gt[pv_i, inst] > 0.5)).sum().item()
    iou = overlap / max(pm_pixels + gt_pixels - overlap, 1)
    print(f"  Instance {inst} (prompt on view {pv_i}):"
          f"  prompt={pm_pixels}px  gt={gt_pixels}px  overlap={overlap}px"
          f"  IoU={iou:.3f}")


## 5. Data Augmentation Effects

Three augmentation categories:
- **Geometric**: image + all masks synchronously transformed
- **Pixel**: only image modified
- **Occlusion**: image filled, gt untouched, prompt mask erased

In [ ]:
# Use the dataset with augmentation enabled
ds_aug = SA1BPuzzlesDataset(
    sa1b_root=SA1B_ROOT,
    shard_names=[shards[0]],
    n_views=6,
    resolution=224,
    max_prompts=4,
    augment=True,
    seed=42,
)

# Non-augmented version (same seed → same sample selection)
ds_noaug = SA1BPuzzlesDataset(
    sa1b_root=SA1B_ROOT,
    shard_names=[shards[0]],
    n_views=6,
    resolution=224,
    max_prompts=4,
    augment=False,
    seed=42,
)

item_raw = ds_noaug[0]
item_aug = ds_aug[0]

print(f"Raw  — imgs: {item_raw['imgs'].shape}, prompt: {item_raw['prompt_masks'].shape}, gt: {item_raw['gt_consistent_masks'].shape}")
print(f"Aug  — imgs: {item_aug['imgs'].shape}, prompt: {item_aug['prompt_masks'].shape}, gt: {item_aug['gt_consistent_masks'].shape}")

# Show side-by-side for first 3 views
n_show = min(3, item_raw["imgs"].shape[0])
fig, axes = plt.subplots(2, n_show, figsize=(5 * n_show, 10))
for v in range(n_show):
    axes[0, v].imshow(item_raw["imgs"][v].permute(1, 2, 0).numpy())
    axes[0, v].set_title(f"Raw — View {v}", fontsize=10)
    axes[0, v].axis("off")

    axes[1, v].imshow(item_aug["imgs"][v].permute(1, 2, 0).numpy())
    axes[1, v].set_title(f"Augmented — View {v}", fontsize=10)
    axes[1, v].axis("off")
plt.suptitle("Raw vs Augmented Images", fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
# Demonstrate individual augmentation types
augmentor_geo = CropAugmentor(geo_prob=1.0, pixel_prob=0.0, occ_prob=0.0)
augmentor_pix = CropAugmentor(geo_prob=0.0, pixel_prob=1.0, occ_prob=0.0)
augmentor_occ = CropAugmentor(geo_prob=0.0, pixel_prob=0.0, occ_prob=1.0)

test_img = (item_raw["imgs"][0].permute(1, 2, 0).numpy() * 255).astype(np.uint8)
test_gt = item_raw["gt_consistent_masks"][0].numpy()   # (M, H, W)
test_pm = item_raw["prompt_masks"][0].numpy()           # (M, H, W)
rng = np.random.default_rng(123)

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
titles = ["Original", "Geometric", "Pixel", "Occlusion"]

def show_col(col, img, gt_m, pm_m, title):
    axes[0, col].imshow(img[:, :, ::-1])
    axes[0, col].set_title(title, fontsize=10)
    axes[0, col].axis("off")
    # Show first GT mask channel
    axes[1, col].imshow(gt_m[0] if gt_m.shape[0] > 0 else np.zeros_like(img[:,:,0]), cmap="Blues", vmin=0, vmax=1)
    axes[1, col].set_title("GT mask [0]", fontsize=9)
    axes[1, col].axis("off")
    # Show first prompt mask channel
    axes[2, col].imshow(pm_m[0] if pm_m.shape[0] > 0 else np.zeros_like(img[:,:,0]), cmap="Oranges", vmin=0, vmax=1)
    axes[2, col].set_title("Prompt mask [0]", fontsize=9)
    axes[2, col].axis("off")

# Original
show_col(0, test_img.copy(), test_gt.copy(), test_pm.copy(), "Original")

# Geometric
img_g, gt_g, pm_g = augmentor_geo(test_img.copy(), test_gt.copy(), test_pm.copy(), rng)
show_col(1, img_g, gt_g, pm_g, "Geometric")

# Pixel
img_p, gt_p, pm_p = augmentor_pix(test_img.copy(), test_gt.copy(), test_pm.copy(), rng)
show_col(2, img_p, gt_p, pm_p, "Pixel")

# Occlusion
img_o, gt_o, pm_o = augmentor_occ(test_img.copy(), test_gt.copy(), test_pm.copy(), rng)
show_col(3, img_o, gt_o, pm_o, "Occlusion")

axes[0, 0].set_ylabel("Image", fontsize=11)
axes[1, 0].set_ylabel("GT mask", fontsize=11)
axes[2, 0].set_ylabel("Prompt mask", fontsize=11)
plt.suptitle("Individual Augmentation Effects (image / GT / prompt)", fontsize=12)
plt.tight_layout()
plt.show()


## 6. Curriculum Scheduling

Parameters change over training progress:
- **Early**: large overlap + large scale (easy samples)
- **Late**: small-to-large overlap + slightly smaller scale (easy+hard mix)

Box scale: 0.45~0.90 (early) → 0.35~0.85 (late)
Overlap: 0.30~0.60 (early) → 0.05~0.55 (late)

In [ ]:
progress_values = np.linspace(0, 1, 50)
min_overlaps, max_overlaps = [], []
min_scales, max_scales = [], []

test_ds = SA1BPuzzlesDataset(
    sa1b_root=SA1B_ROOT,
    shard_names=[shards[0]],
    n_views=6,
    resolution=224,
    augment=False,
    seed=42,
)

for p in progress_values:
    test_ds.set_curriculum(float(p))
    min_overlaps.append(test_ds.puzzle_kwargs["min_overlap"])
    max_overlaps.append(test_ds.puzzle_kwargs["max_overlap"])
    min_scales.append(test_ds.puzzle_kwargs["min_box_size_scale"])
    max_scales.append(test_ds.puzzle_kwargs["max_box_size_scale"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].fill_between(progress_values, min_overlaps, max_overlaps, alpha=0.3, color="C0")
axes[0].plot(progress_values, min_overlaps, "C0-", label="min_overlap")
axes[0].plot(progress_values, max_overlaps, "C0--", label="max_overlap")
axes[0].set_xlabel("Training Progress")
axes[0].set_ylabel("Overlap")
axes[0].set_title("Overlap Curriculum\n(upper stays high = easy samples always present)")
axes[0].legend()
axes[0].set_ylim(0, 0.7)
axes[0].grid(True, alpha=0.3)

axes[1].fill_between(progress_values, min_scales, max_scales, alpha=0.3, color="C1")
axes[1].plot(progress_values, min_scales, "C1-", label="min_box_size_scale")
axes[1].plot(progress_values, max_scales, "C1--", label="max_box_size_scale")
axes[1].set_xlabel("Training Progress")
axes[1].set_ylabel("Box Size Scale")
axes[1].set_title("Box Scale Curriculum\n(0.45~0.90 -> 0.35~0.85)")
axes[1].legend()
axes[1].set_ylim(0, 1.0)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Early  (p=0.0): overlap=[{min_overlaps[0]:.2f}, {max_overlaps[0]:.2f}]  scale=[{min_scales[0]:.2f}, {max_scales[0]:.2f}]")
print(f"Late   (p=1.0): overlap=[{min_overlaps[-1]:.2f}, {max_overlaps[-1]:.2f}]  scale=[{min_scales[-1]:.2f}, {max_scales[-1]:.2f}]")


## 7. Multi-Sample Browser

Browse 8 diverse samples from different shards and offsets.
For each sample, two figures are shown:
1. **Raw** — Image / Prompt / GT (no augmentation)
2. **Augmented** — the same sample after geometric, pixel, and occlusion augmentation

Same color = same physical object across prompt and GT.

In [ ]:
PALETTE = [
    (0.90, 0.30, 0.30), (0.30, 0.80, 0.30), (0.30, 0.50, 0.90),
    (0.95, 0.75, 0.20), (0.80, 0.30, 0.80), (0.20, 0.85, 0.85),
]

def overlay_masks(img_np, masks, palette):
    ov = img_np.copy()
    M = masks.shape[0]
    for m in range(M):
        mask = masks[m].numpy() > 0.5
        if mask.any():
            c = np.array(palette[m % len(palette)])
            ov[mask] = ov[mask] * 0.35 + c * 0.65
    return ov

def plot_sample(item, axes_3xN):
    imgs_s = item["imgs"]
    gt_s = item["gt_consistent_masks"]
    pm_s = item["prompt_masks"]
    pv_s = item["prompt_views"]
    N = imgs_s.shape[0]
    M = pm_s.shape[1]
    pv_set = set(pv_s.numpy().tolist())

    for v in range(N):
        img_np = imgs_s[v].permute(1, 2, 0).numpy()
        axes_3xN[0, v].imshow(img_np)
        t = f"V{v}" + (" *" if v in pv_set else "")
        axes_3xN[0, v].set_title(t, fontsize=8)
        axes_3xN[0, v].axis("off")

        axes_3xN[1, v].imshow(overlay_masks(img_np, pm_s[v], PALETTE))
        axes_3xN[1, v].set_title("Prompt" if pm_s[v].sum() > 0 else "empty", fontsize=7)
        axes_3xN[1, v].axis("off")

        axes_3xN[2, v].imshow(overlay_masks(img_np, gt_s[v], PALETTE))
        axes_3xN[2, v].set_title(f"GT ({M})", fontsize=7)
        axes_3xN[2, v].axis("off")

# Build raw + augmented datasets (same seed = same sample selection)
ds_raw = SA1BPuzzlesDataset(
    sa1b_root=SA1B_ROOT, n_views=6, resolution=224,
    max_prompts=4, augment=False, seed=42,
)
ds_aug = SA1BPuzzlesDataset(
    sa1b_root=SA1B_ROOT, n_views=6, resolution=224,
    max_prompts=4, augment=True, seed=42,
)
total = len(ds_raw)
print(f"Total dataset size: {total} samples")
print(f"Default bbox scale: [{ds_raw.puzzle_kwargs['min_box_size_scale']:.2f}, {ds_raw.puzzle_kwargs['max_box_size_scale']:.2f}]")
print(f"Default overlap:    [{ds_raw.puzzle_kwargs['min_overlap']:.2f}, {ds_raw.puzzle_kwargs['max_overlap']:.2f}]")
print()

n_browse = 16
indices = np.linspace(0, total - 1, n_browse, dtype=int).tolist()
print(f"Browsing indices: {indices}\n")

for s_idx in indices:
    item_raw = ds_raw[s_idx]
    item_aug = ds_aug[s_idx]
    M = item_raw["prompt_masks"].shape[1]
    pv = item_raw["prompt_views"].tolist()
    N = item_raw["imgs"].shape[0]

    assert item_raw["prompt_masks"].shape[1] == item_raw["gt_consistent_masks"].shape[1]

    fig, axes = plt.subplots(6, N, figsize=(2.8 * N, 16))

    plot_sample(item_raw, axes[0:3])
    plot_sample(item_aug, axes[3:6])

    axes[0, 0].set_ylabel("Raw\nImage", fontsize=9)
    axes[1, 0].set_ylabel("Raw\nPrompt", fontsize=9)
    axes[2, 0].set_ylabel("Raw\nGT", fontsize=9)
    axes[3, 0].set_ylabel("Aug\nImage", fontsize=9, color="purple")
    axes[4, 0].set_ylabel("Aug\nPrompt", fontsize=9, color="purple")
    axes[5, 0].set_ylabel("Aug\nGT", fontsize=9, color="purple")

    scale_lo = ds_raw.puzzle_kwargs["min_box_size_scale"]
    scale_hi = ds_raw.puzzle_kwargs["max_box_size_scale"]
    ov_lo = ds_raw.puzzle_kwargs["min_overlap"]
    ov_hi = ds_raw.puzzle_kwargs["max_overlap"]

    plt.suptitle(
        f"Sample #{s_idx}: {M} inst, views={pv}, 1:1=True"
        f"  |  scale=[{scale_lo:.2f},{scale_hi:.2f}]  overlap=[{ov_lo:.2f},{ov_hi:.2f}]",
        fontsize=10, fontweight="bold",
    )
    plt.tight_layout()
    plt.show()
    print()


## 8. Summary: Pipeline Feature Status

Overview of which features are active in the current training pipeline.

In [ ]:
features = [
    ("1:1 Prompt-GT correspondence", "Active", "prompt[:, i] and gt[:, i] same object, M_prompt == M_gt"),
    ("Single-view prompt propagation", "Active", "_build_prompt_masks: random view per prompt"),
    ("Multi-prompt support", "Active", "max_prompts=4, each on random view"),
    ("Boundary noise on prompts", "Active", "erode/dilate with prompt_boundary_noise=0.3"),
    ("Direct per-channel loss", "Active", "No Hungarian matching - direct BCE+Dice on aligned channels"),
    ("Curriculum overlap scheduling", "Active (--curriculum)", "set_curriculum(progress) adjusts overlap & scale"),
    ("Geometric augmentation", "Active", "flip, rotate, affine, perspective, resized crop"),
    ("Pixel augmentation", "Active", "color jitter, grayscale, blur, sharpen, JPEG, noise"),
    ("Occlusion augmentation", "Active", "random erasing, grid mask, coarse dropout"),
    ("GT mask occlusion handling", "Untouched", "GT stays complete for completion learning"),
    ("Prompt mask occlusion handling", "Erased", "Simulates SAM seeing partial object"),
    ("Hungarian matching", "Removed", "Not needed - prompt and GT are 1:1 aligned"),
    ("Mosaic / MixUp / CutMix", "Disabled", "Would break cross-view consistency"),
    ("Copy-Paste", "Disabled", "Would disrupt GT consistency IDs"),
    ("Per-view SAM noise simulation", "Removed", "Replaced by single-view prompt paradigm"),
]

header = f"{'Feature':<40s} {'Status':<25s} Notes"
print(header)
print("-" * 105)
for feat, status, notes in features:
    print(f"{feat:<40s} {status:<25s} {notes}")
